In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from rdkit import Chem
from rdkit.Chem import Descriptors
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

def get_descriptors(df, set_name):
    """Helper function to calculate chemical properties."""
    print(f"Processing {set_name}...")
    # Normalize column names to lowercase
    df.columns = [c.lower() for c in df.columns]
    
    if 'smiles' not in df.columns:
        print(f"Error: 'smiles' column not found in {set_name} data.")
        return pd.DataFrame()

    mols = [Chem.MolFromSmiles(smi) for smi in df['smiles']]
    df['MW'] = [Descriptors.MolWt(mol) if mol else None for mol in mols]
    df['LogP'] = [Descriptors.MolLogP(mol) if mol else None for mol in mols]
    df['HBD'] = [Descriptors.NumHDonors(mol) if mol else None for mol in mols]
    df['HBA'] = [Descriptors.NumHAcceptors(mol) if mol else None for mol in mols]
    df['Set'] = set_name
    # Remove rows where RDKit failed to parse SMILES
    return df.dropna(subset=['MW', 'LogP', 'HBD', 'HBA'])

# 1. Load and Process all three datasets
train_df = get_descriptors(pd.read_csv('EZH2_train_set.csv'), 'Train')
test_df = get_descriptors(pd.read_csv('EZH2_test_set.csv'), 'Test')
external_df = get_descriptors(pd.read_csv('externaldata_ezh2_dataset_classified.csv'), 'External')

# Combine all data for consistent scaling and PCA
combined = pd.concat([train_df, test_df, external_df], axis=0).reset_index(drop=True)

# 2. PCA Analysis
# Define descriptors for PCA
features = ['MW', 'LogP', 'HBD', 'HBA']
x = combined[features].values

# Standardize the features (Mean=0, Variance=1) - Critical for PCA
x_scaled = StandardScaler().fit_transform(x)

# Perform PCA
pca = PCA(n_components=2)
principalComponents = pca.fit_transform(x_scaled)
pca_df = pd.DataFrame(data=principalComponents, columns=['PC1', 'PC2'])
pca_df['Set'] = combined['Set']

# 3. Visualization: PCA Plot
plt.figure(figsize=(10, 7))
sns.scatterplot(
    data=pca_df, 
    x='PC1', 
    y='PC2', 
    hue='Set', 
    style='Set', 
    alpha=0.5, 
    palette='viridis', 
    s=70
)

# Add variance explained ratio to labels
var_exp = pca.explained_variance_ratio_
plt.xlabel(f'Principal Component 1 ({var_exp[0]:.2%} Variance)')
plt.ylabel(f'Principal Component 2 ({var_exp[1]:.2%} Variance)')
plt.title('PCA of Chemical Space: Train vs Test vs External Set')
plt.grid(True, linestyle='--', alpha=0.5)
plt.savefig('EZH2_PCA_Comparison_3sets_new.png', dpi=600)

# 4. Visualization: PairPlot
pair_plot = sns.pairplot(combined[features + ['Set']], hue='Set', diag_kind='kde', palette='bright', plot_kws={'alpha':0.4})
pair_plot.savefig('Descriptor_PairPlot_3sets_new.png', dpi=300)

plt.show()

print("All plots generated successfully.")
print(f"PCA Total Variance Explained: {sum(var_exp):.2%}")